# gigapath_runpod_modeling (S3 features → RunPod MIL 학습)

`gigapath_runpod.ipynb`에서 임베딩(.pt)을 생성해 S3에 올렸다고 가정하고, RunPod에서 다운로드→MIL 학습까지 이어서 실행하는 노트북입니다. Attention MIL / CLAM 스타일 / TransMIL을 모두 지원하는 단일 스크립트를 원격에 작성·실행합니다.


## 0. 기본 설정
- RunPod SSH/rsync 접속 정보와 경로(기존 노트북과 동일 기본값)
- S3 접두사(`S3_PREFIX`)와 AWS 프로파일/리전
- 라벨 parquet 업로드 후 manifest.csv를 원격에서 생성(split 포함)
- 토글: `RUN_REMOTE_SETUP`, `RUN_S3_SYNC`, `WRITE_TRAINER`, `RUN_TRAINING`


In [1]:

import os, shlex, subprocess, time, json
from pathlib import Path

# SSH 대상 및 키 (기존 runpod 설정 재사용)
SSH_HOST_GATEWAY = 'v4wn4o1spmckrh-64410ec0@ssh.runpod.io'
SSH_HOST_DIRECT = 'root@64.247.206.80'
SSH_PORT_DIRECT = 46258
SSH_KEY = '~/.ssh/runpod_peter'
USE_DIRECT_FOR_SSH = True
USE_DIRECT_FOR_RSYNC = True
SSH_EXTRA_OPTS = ''

# RunPod 경로
REMOTE_BASE = '/workspace/data'
REMOTE_RAW = f"{REMOTE_BASE}/raw"
REMOTE_WORK = f"{REMOTE_BASE}/work"
REMOTE_FEATURE_DIR = f"{REMOTE_WORK}/features"
REMOTE_CHECKPOINT = f"{REMOTE_WORK}/checkpoints"
REMOTE_LOG = f"{REMOTE_BASE}/logs"
REMOTE_MANIFEST = f"{REMOTE_WORK}/manifest.csv"
REMOTE_TRAIN_SCRIPT = f"{REMOTE_WORK}/train_mil.py"
REMOTE_VENV = '~/venv_gigapath'

# S3 설정 (임베딩 .pt 저장 위치)
S3_PREFIX = 's3://gc-pathology/gv-level1-embedding/'  # 접두사만 작성
AWS_PROFILE = None  # 예: 'default' 또는 None
AWS_REGION = 'ap-northeast-2'
AWS_ENDPOINT = None  # 커스텀 엔드포인트 있으면 입력
RUN_S3_SYNC = True

# 라벨/manifest 설정 (필요 시 경로 수정)
LABEL_PARQUET_LOCAL = Path('/Users/curv/Repos/GC-Pathology/PoC/v1/mammary_adenoma_vs_adenocarcinoma_only(2023).parquet')
LABEL_MAP = {"adenoma": 0, "adenocarcinoma": 1, "mammary_adenoma": 0, "mammary_adenocarcinoma": 1}
VAL_RATIO = 0.15
TEST_RATIO = 0.0
RANDOM_SEED = 42
MAX_TILES_PER_SLIDE = 8000  # 메모리 안전용 샘플링 상한 (None이면 전체)

# 학습/스크립트 토글
RUN_REMOTE_SETUP = True
WRITE_TRAINER = True
RUN_TRAINING = True

print('SSH_HOST_DIRECT=', SSH_HOST_DIRECT, 'port', SSH_PORT_DIRECT)
print('REMOTE_FEATURE_DIR=', REMOTE_FEATURE_DIR)
print('S3_PREFIX=', S3_PREFIX)
print('워크플로우: 1) 원격 디렉터리 준비 → 2) venv 설치 → 3) S3 동기화 → 4) manifest 생성 → 5) MIL 학습/평가')


SSH_HOST_DIRECT= root@64.247.206.80 port 46258
REMOTE_FEATURE_DIR= /workspace/data/work/features
S3_PREFIX= s3://gc-pathology/gv-level1-embedding/
워크플로우: 1) 원격 디렉터리 준비 → 2) venv 설치 → 3) S3 동기화 → 4) manifest 생성 → 5) MIL 학습/평가


## 1. 로컬→RunPod 유틸리티 (SSH/rsync)
- `run_local`, `run_ssh`, `rsync_upload/download`
- 원격 경로 생성/상태 확인 헬퍼 포함


In [2]:

def run_local(cmd, check=True):
    print(f"[local] $ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        msg = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(f"Local command failed ({result.returncode}): {cmd}{msg}")
    return result.returncode


def _ssh_parts(host, port=None):
    parts = ['ssh']
    if SSH_KEY:
        parts += ['-i', SSH_KEY]
    if SSH_EXTRA_OPTS:
        parts += shlex.split(SSH_EXTRA_OPTS)
    if port:
        parts += ['-p', str(port)]
    parts += [host]
    return parts


def run_ssh(cmd, check=True, use_direct=None):
    if use_direct is None:
        use_direct = USE_DIRECT_FOR_SSH
    parts = _ssh_parts(
        SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY,
        SSH_PORT_DIRECT if use_direct else None,
    )
    ssh_cmd = ' '.join(shlex.quote(p) for p in parts + [cmd])
    print(f"[ssh] $ {cmd}")
    result = subprocess.run(ssh_cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        msg = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(f"SSH command failed ({result.returncode}): {cmd}{msg}")
    return result.returncode


def _rsync_ssh_opt(use_direct=True):
    key = Path(SSH_KEY).expanduser() if SSH_KEY else None
    parts = ['ssh']
    if key:
        parts += ['-i', str(key)]
    if SSH_EXTRA_OPTS:
        parts += shlex.split(SSH_EXTRA_OPTS)
    if use_direct:
        parts += ['-p', str(SSH_PORT_DIRECT)]
    return ' '.join(parts)


def rsync_upload(local_path: Path, remote_dir: str):
    use_direct = USE_DIRECT_FOR_RSYNC
    ssh_opt = _rsync_ssh_opt(use_direct)
    remote_host = SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY
    remote_target = f"{remote_host}:{remote_dir.rstrip('/')}/"
    cmd = (
        f"rsync -avh --no-perms --no-owner --no-group --partial --progress -e {shlex.quote(ssh_opt)} "
        f"{shlex.quote(str(local_path))} {remote_target}"
    )
    run_local(cmd)


def rsync_download(remote_path: str, local_dir: Path):
    use_direct = USE_DIRECT_FOR_RSYNC
    ssh_opt = _rsync_ssh_opt(use_direct)
    remote_host = SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY
    local_dir.mkdir(parents=True, exist_ok=True)
    remote_src = f"{remote_host}:{remote_path}"
    cmd = (
        f"rsync -avh --no-perms --no-owner --no-group --partial --progress -e {shlex.quote(ssh_opt)} "
        f"{remote_src} {shlex.quote(str(local_dir))}/"
    )
    run_local(cmd)


def ensure_remote_dirs():
    print('[단계] 원격 디렉터리 및 GPU 상태 확인 중...')
    run_ssh(f"mkdir -p {REMOTE_FEATURE_DIR} {REMOTE_CHECKPOINT} {REMOTE_LOG} {REMOTE_WORK}")
    run_ssh("echo 'SSH OK on $(hostname)' && nvidia-smi || true", check=False)
    run_ssh("df -h . || true", check=False)
    print('[완료] 원격 디렉터리 준비 완료')


ensure_remote_dirs()


[단계] 원격 디렉터리 및 GPU 상태 확인 중...
[ssh] $ mkdir -p /workspace/data/work/features /workspace/data/work/checkpoints /workspace/data/logs /workspace/data/work
[ssh] $ echo 'SSH OK on $(hostname)' && nvidia-smi || true
SSH OK on $(hostname)
Mon Dec  1 02:43:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40                     On  |   00000000:81:00.0 Off |                    0 |

## 2. 원격 의존성 설치 (venv)
- torch는 RunPod 기본 CUDA 이미지에 있다고 가정, 부족 시 버전 명시 가능
- 추가 패키지: scikit-learn, torchmetrics, einops, awscli 등


In [3]:

REMOTE_PY_PKGS = [
    'pip', 'setuptools', 'wheel',
    'pandas', 'numpy', 'scikit-learn', 'tqdm', 'torchmetrics', 'einops',
    'pyyaml', 'awscli', 'pyarrow'
]

RUN_REMOTE_SETUP = bool(RUN_REMOTE_SETUP)


def install_remote_deps(pkgs=None, venv_path=REMOTE_VENV):
    pkgs = pkgs or REMOTE_PY_PKGS
    pkg_str = ' '.join(pkgs)
    cmd = (
        f"python3 -m venv {venv_path} && "
        f"{venv_path}/bin/pip install --upgrade pip setuptools wheel && "
        f"{venv_path}/bin/pip install {pkg_str}"
    )
    print('[단계] 가상환경 설치/업데이트 시작 ->', venv_path)
    print('       패키지:', pkg_str)
    run_ssh(cmd)
    print('[완료] 가상환경 패키지 설치 완료')


if RUN_REMOTE_SETUP:
    install_remote_deps()
else:
    print('가상환경 설치 건너뜀 (RUN_REMOTE_SETUP=False)')


[단계] 가상환경 설치/업데이트 시작 -> ~/venv_gigapath
       패키지: pip setuptools wheel pandas numpy scikit-learn tqdm torchmetrics einops pyyaml awscli pyarrow
[ssh] $ python3 -m venv ~/venv_gigapath && ~/venv_gigapath/bin/pip install --upgrade pip setuptools wheel && ~/venv_gigapath/bin/pip install pip setuptools wheel pandas numpy scikit-learn tqdm torchmetrics einops pyyaml awscli pyarrow

[완료] 가상환경 패키지 설치 완료


## 3. S3 → RunPod feature 동기화
- `RUN_S3_SYNC=True`로 토글 후 실행
- AWS 자격 증명은 RunPod에 미리 export하거나 `AWS_PROFILE`/`AWS_*` 값으로 전달


In [4]:

RUN_S3_SYNC = bool(RUN_S3_SYNC)


def sync_s3_features():
    env_exports = []
    if AWS_PROFILE:
        env_exports.append(f"AWS_PROFILE={AWS_PROFILE}")
    if AWS_REGION:
        env_exports.append(f"AWS_DEFAULT_REGION={AWS_REGION}")
    if AWS_ENDPOINT:
        env_exports.append(f"AWS_ENDPOINT_URL={AWS_ENDPOINT}")
    env_prefix = ' '.join(env_exports) + ' ' if env_exports else ''
    cmd = f"{env_prefix}{REMOTE_VENV}/bin/aws s3 sync {S3_PREFIX} {REMOTE_FEATURE_DIR}/"
    print(f"[단계] S3 임베딩 동기화 시작: {S3_PREFIX} -> {REMOTE_FEATURE_DIR}")
    run_ssh(cmd)
    stat_cmd = (
        f"echo '[remote] 동기화 후 PT 파일 개수:' && find {REMOTE_FEATURE_DIR} -maxdepth 1 -name '*.pt' | wc -l && "
        f"du -sh {REMOTE_FEATURE_DIR} || true; "
        f"ls -1 {REMOTE_FEATURE_DIR} | head || true; "
        f"ls -1 {REMOTE_FEATURE_DIR} | tail || true"
    )
    run_ssh(stat_cmd, check=False)
    print('[완료] S3 동기화 완료')


if RUN_S3_SYNC:
    sync_s3_features()
else:
    print('S3 sync skipped (RUN_S3_SYNC=False)')


[단계] S3 임베딩 동기화 시작: s3://gc-pathology/gv-level1-embedding/ -> /workspace/data/work/features
[ssh] $ AWS_DEFAULT_REGION=ap-northeast-2 ~/venv_gigapath/bin/aws s3 sync s3://gc-pathology/gv-level1-embedding/ /workspace/data/work/features/
[ssh] $ echo '[remote] 동기화 후 PT 파일 개수:' && find /workspace/data/work/features -maxdepth 1 -name '*.pt' | wc -l && du -sh /workspace/data/work/features || true; ls -1 /workspace/data/work/features | head || true; ls -1 /workspace/data/work/features | tail || true
[remote] 동기화 후 PT 파일 개수:
48
4.2M	/workspace/data/work/features
S23-00026#1###9.pt
S23-00080#1###1.pt
S23-00233#1###9.pt
S23-00449#1###0.pt
S23-00525#1###4.pt
S23-00963#1###7.pt
S23-01275#1###7.pt
S23-01304#1###6.pt
S23-01340#1###6.pt
S23-01481#1###5.pt
S23-07953#1###7.pt
S23-08020#1###4.pt
S23-08066#1###2.pt
S23-08129#1###2.pt
S23-08235#1###3.pt
S23-08797#1###8.pt
S23-08939#1###3.pt
S23-08979#1###5.pt
S23-09017#1###9.pt
S23-09193#1###9.pt

[완료] S3 동기화 완료


## 4. 라벨 parquet 업로드 및 manifest 생성
- 로컬 parquet(`LABEL_PARQUET_LOCAL`)을 rsync 업로드 후 원격에서 manifest.csv를 만듭니다.
- 라벨 컬럼 소문자 매핑(`LABEL_MAP`)을 사용, `FILE_NAME`/`INSP_RQST_NO`/`FOLDER` 중 일치하는 키로 매칭.
- `VAL_RATIO`/`TEST_RATIO`로 split 컬럼 추가.


In [5]:

from string import Template

manifest_template = Template(r'''
$REMOTE_VENV/bin/python - <<'PY'
import json, random
from collections import Counter
from pathlib import Path
import subprocess

try:
    import pandas as pd
except ImportError:
    print('[단계] pandas/pyarrow 미설치 → 설치 시도')
    subprocess.check_call(['$REMOTE_VENV/bin/pip', 'install', '-q', 'pandas', 'pyarrow'])
    import pandas as pd

print('[단계] 라벨 parquet와 임베딩을 이용해 manifest 생성')

label_path = Path('$REMOTE_WORK') / '$LABEL_NAME'
features_dir = Path('$REMOTE_FEATURE_DIR')
manifest_path = Path('$REMOTE_MANIFEST')
label_map = $LABEL_MAP_JSON
val_ratio = $VAL_RATIO
test_ratio = $TEST_RATIO
seed = $RANDOM_SEED

print('  label parquet:', label_path)
print('  feature dir:', features_dir)
print('  split 비율 -> val', val_ratio, 'test', test_ratio)

if not label_path.exists():
    raise SystemExit(f'label parquet missing on remote: {label_path}')
if not features_dir.exists():
    raise SystemExit(f'feature dir missing: {features_dir}')

df = pd.read_parquet(label_path)

label_col = None
for cand in ['label', 'LABEL']:
    if cand in df.columns:
        label_col = cand
        break
if label_col is None:
    raise SystemExit('label column not found; expected "label"')

print('라벨 분포 (parquet):', df[label_col].value_counts().to_dict())


def norm_label(x):
    key = str(x).lower().strip()
    return label_map.get(key)


def candidate_ids(row):
    ids = []
    file_names = str(row.get('FILE_NAME', '')).split('|')
    for fn in file_names:
        fn = fn.strip()
        if fn:
            ids.append(Path(fn).stem)
    slide_id = row.get('INSP_RQST_NO', row.get('INSP_RQST_NUM', row.get('FOLDER', '')))
    slide_id = str(slide_id).strip()
    if slide_id:
        ids.append(slide_id)
    return [i for i in ids if i]

label_index = {}
for _, row in df.iterrows():
    lbl = norm_label(row[label_col])
    if lbl is None:
        continue
    for cid in candidate_ids(row):
        label_index[cid] = int(lbl)

print('라벨 인덱스 매핑 개수:', len(label_index))

feature_files = sorted(features_dir.glob('*.pt'))
print('다운로드된 feature 파일 개수:', len(feature_files))

records = []
unmatched = []
for pt in feature_files:
    slide_id = pt.stem.replace('_gigapath', '')
    lbl = label_index.get(slide_id)
    if lbl is None:
        unmatched.append(slide_id)
        continue
    records.append({'slide_id': slide_id, 'label': lbl, 'pt_path': str(pt.resolve())})

if not records:
    raise SystemExit('No features matched labels; check naming conventions.')

random.seed(seed)
random.shuffle(records)

val_n = int(len(records) * val_ratio)
test_n = int(len(records) * test_ratio)
for i, rec in enumerate(records):
    if i < val_n:
        rec['split'] = 'val'
    elif i < val_n + test_n:
        rec['split'] = 'test'
    else:
        rec['split'] = 'train'

man_df = pd.DataFrame.from_records(records)
print('매칭된 라벨 분포:', dict(Counter(man_df['label'])))
print('split 개수:', man_df['split'].value_counts().to_dict())
man_df.to_csv(manifest_path, index=False)
print('manifest 저장 완료:', manifest_path)
if unmatched:
    print('라벨 매칭 실패 feature (앞 10개):', unmatched[:10])
PY
''')


def upload_labels():
    if not LABEL_PARQUET_LOCAL.exists():
        raise FileNotFoundError(f'label parquet not found: {LABEL_PARQUET_LOCAL}')
    print(f"[단계] 라벨 parquet 업로드: {LABEL_PARQUET_LOCAL.name} -> {REMOTE_WORK}")
    rsync_upload(LABEL_PARQUET_LOCAL, REMOTE_WORK)
    print('[완료] 라벨 업로드 완료')


def build_remote_manifest():
    print('[단계] 원격 manifest 생성 시작')
    script = manifest_template.substitute(
        REMOTE_WORK=REMOTE_WORK,
        LABEL_NAME=LABEL_PARQUET_LOCAL.name,
        REMOTE_FEATURE_DIR=REMOTE_FEATURE_DIR,
        REMOTE_MANIFEST=REMOTE_MANIFEST,
        LABEL_MAP_JSON=json.dumps({k: int(v) for k, v in LABEL_MAP.items()}),
        VAL_RATIO=VAL_RATIO,
        TEST_RATIO=TEST_RATIO,
        RANDOM_SEED=RANDOM_SEED,
        REMOTE_VENV=REMOTE_VENV,
    )
    run_ssh(script)
    print('[완료] 원격 manifest 생성 완료')


upload_labels()
build_remote_manifest()


[단계] 라벨 parquet 업로드: mammary_adenoma_vs_adenocarcinoma_only(2023).parquet -> /workspace/data/work
[local] $ rsync -avh --no-perms --no-owner --no-group --partial --progress -e 'ssh -i /Users/curv/.ssh/runpod_peter -p 46258' '/Users/curv/Repos/GC-Pathology/PoC/v1/mammary_adenoma_vs_adenocarcinoma_only(2023).parquet' root@64.247.206.80:/workspace/data/work/
Transfer starting: 1 files

sent 90 bytes  received 20 bytes  214 bytes/sec
total size is 1969k  speedup is 17901.16

[완료] 라벨 업로드 완료
[단계] 원격 manifest 생성 시작
[ssh] $ 
~/venv_gigapath/bin/python - <<'PY'
import json, random
from collections import Counter
from pathlib import Path
import subprocess

try:
    import pandas as pd
except ImportError:
    print('[단계] pandas/pyarrow 미설치 → 설치 시도')
    subprocess.check_call(['~/venv_gigapath/bin/pip', 'install', '-q', 'pandas', 'pyarrow'])
    import pandas as pd

print('[단계] 라벨 parquet와 임베딩을 이용해 manifest 생성')

label_path = Path('/workspace/data/work') / 'mammary_adenoma_vs_adenocarcinoma_only(2

## 5. 원격 MIL 학습 스크립트 작성 (Attention/CLAM/TransMIL)
- RunPod에 `train_mil.py`를 생성합니다.
- 배치 단위=슬라이드 1개(bag), AMP 지원, pos-weight 자동 계산.
- 입력 manifest: `slide_id,label,pt_path,split` 컬럼.


In [6]:

TRAIN_SCRIPT = r'''
import argparse, json, os, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score, confusion_matrix

FEATURE_KEYS = ['features', 'feats', 'embedding', 'embeddings', 'feat']
COORD_KEYS = ['coords', 'coord', 'locs', 'locations']

def load_feats_coords(data):
    import re
    # 우선순위: 명시적인 키 -> last_layer_embed -> layer_<n>_embed 중 가장 높은 레이어
    feature_keys = ['features', 'feats', 'embedding', 'embeddings', 'feat', 'last_layer_embed']
    feats = None
    for k in feature_keys:
        if k in data:
            feats = data[k]
            break
    if feats is None:
        layer_keys = [k for k in data.keys() if re.match(r'^layer_\d+_embed$', k)]
        if layer_keys:
            best = sorted(layer_keys, key=lambda x: int(re.findall(r'\d+', x)[0]))[-1]
            feats = data[best]
    coords = None
    for k in ['coords', 'coord', 'locs', 'locations']:
        if k in data:
            coords = data[k]
            break
    if feats is None:
        raise KeyError(f"feature key not found in pt; available keys={list(data.keys())}")
    return feats, coords


from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from einops import rearrange


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {}
    try:
        metrics['auc'] = float(roc_auc_score(y_true, y_prob))
    except Exception:
        metrics['auc'] = float('nan')
    try:
        metrics['ap'] = float(average_precision_score(y_true, y_prob))
    except Exception:
        metrics['ap'] = float('nan')
    metrics['f1'] = float(f1_score(y_true, y_pred)) if len(y_true) else float('nan')
    metrics['acc'] = float(accuracy_score(y_true, y_pred)) if len(y_true) else float('nan')
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    except Exception:
        tn = fp = fn = tp = float('nan')
    metrics['confusion'] = {'tn': float(tn), 'fp': float(fp), 'fn': float(fn), 'tp': float(tp)}
    return metrics


class SlideBagDataset(Dataset):
    def __init__(self, manifest_path: Path, split: str, max_tiles=None, shuffle_tiles=False):
        df = pd.read_csv(manifest_path)
        self.df = df[df['split'] == split].reset_index(drop=True)
        self.max_tiles = max_tiles
        self.shuffle_tiles = shuffle_tiles

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        data = torch.load(row['pt_path'], map_location='cpu')
        feats, coords = load_feats_coords(data)
        if isinstance(coords, np.ndarray):
            coords = torch.from_numpy(coords)
        feats = feats.float()
        if coords is not None:
            coords = coords.int()
        if self.shuffle_tiles:
            perm = torch.randperm(len(feats))
            feats = feats[perm]
            if coords is not None:
                coords = coords[perm]
        if self.max_tiles and len(feats) > self.max_tiles:
            idxs = torch.randperm(len(feats))[: self.max_tiles]
            feats = feats[idxs]
            if coords is not None:
                coords = coords[idxs]
        label = torch.tensor(float(row['label']), dtype=torch.float32)
        return feats, coords, label, row['slide_id']


def collate_fn(batch):
    feats, coords, labels, slide_ids = zip(*batch)
    labels = torch.tensor(labels, dtype=torch.float32)
    return list(feats), list(coords), labels, list(slide_ids)


class AttentionMIL(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, dropout=0.2, gated=False):
        super().__init__()
        self.gated = gated
        self.norm = nn.LayerNorm(in_dim)
        self.dropout = nn.Dropout(dropout)
        self.attn_v = nn.Linear(in_dim, hidden_dim)
        self.attn_u = nn.Linear(in_dim, hidden_dim) if gated else None
        self.attn_out = nn.Linear(hidden_dim, 1)
        self.classifier = nn.Linear(in_dim, 1)

    def forward(self, feats, coords=None):
        x = self.dropout(self.norm(feats))
        v = torch.tanh(self.attn_v(x))
        if self.gated:
            u = torch.sigmoid(self.attn_u(x))
            v = v * u
        attn_score = self.attn_out(v).squeeze(-1)  # (N,)
        weight = torch.softmax(attn_score, dim=0)
        pooled = torch.sum(weight.unsqueeze(-1) * x, dim=0)
        logit = self.classifier(pooled).squeeze(0)
        return logit, weight


class TransMIL(nn.Module):
    def __init__(self, in_dim, depth=2, heads=4, ff_dim=512, dropout=0.1, use_pos_enc=False):
        super().__init__()
        self.use_pos_enc = use_pos_enc
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=in_dim, nhead=heads, dim_feedforward=ff_dim, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, in_dim))
        self.cls_bias = nn.Parameter(torch.zeros(1, 1, in_dim))
        self.pos_mlp = nn.Sequential(nn.LayerNorm(2), nn.Linear(2, in_dim)) if use_pos_enc else None
        self.classifier = nn.Linear(in_dim, 1)

    def forward(self, feats, coords=None):
        x = feats.unsqueeze(0)  # (1, N, D)
        if self.use_pos_enc and coords is not None:
            pos = coords.float()
            pos = (pos - pos.mean(0, keepdim=True)) / (pos.std(0, keepdim=True) + 1e-6)
            x = x + self.pos_mlp(pos).unsqueeze(0)
        cls = self.cls_token + self.cls_bias
        tokens = torch.cat([cls, x], dim=1)
        out = self.encoder(tokens)
        cls_out = out[:, 0]  # (B, D)
        logit = self.classifier(cls_out).squeeze(1)
        # attn: (B, N, D) @ (B, D, 1) -> (B, N, 1) -> (B, N)
        attn_score = torch.matmul(out[:, 1:], cls_out.unsqueeze(-1)).squeeze(-1)
        weight = torch.softmax(attn_score, dim=-1)
        return logit.squeeze(0), weight.squeeze(0)


def infer_in_dim(manifest_path: Path):
    df = pd.read_csv(manifest_path)
    if df.empty:
        raise RuntimeError('manifest is empty')
    sample_pt = Path(df.iloc[0]['pt_path'])
    data = torch.load(sample_pt, map_location='cpu')
    feats, _ = load_feats_coords(data)
    return int(feats.shape[1])


def make_loaders(manifest_path, max_tiles, num_workers):
    train_ds = SlideBagDataset(manifest_path, 'train', max_tiles=max_tiles, shuffle_tiles=True)
    val_ds = SlideBagDataset(manifest_path, 'val', max_tiles=max_tiles, shuffle_tiles=False)
    test_ds = SlideBagDataset(manifest_path, 'test', max_tiles=max_tiles, shuffle_tiles=False)

    def _loader(ds, shuffle):
        if len(ds) == 0:
            return None
        return DataLoader(ds, batch_size=1, shuffle=shuffle, num_workers=num_workers, collate_fn=collate_fn)

    return _loader(train_ds, True), _loader(val_ds, False), _loader(test_ds, False), train_ds, val_ds, test_ds


def train(args):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    set_seed(args.seed)

    manifest = pd.read_csv(args.manifest)
    split_counts = manifest['split'].value_counts().to_dict()
    label_counts = manifest['label'].value_counts().to_dict()
    print(f"[단계] manifest 로드 ({args.manifest}) -> 총 {len(manifest)} 슬라이드")
    print('       split 분포:', split_counts)
    print('       라벨 분포:', label_counts)

    in_dim = infer_in_dim(args.manifest)
    if args.model == 'transmil':
        model = TransMIL(
            in_dim,
            depth=args.depth,
            heads=args.heads,
            ff_dim=args.ffn_dim,
            dropout=args.dropout,
            use_pos_enc=args.use_pos_enc,
        )
    elif args.model == 'clam':
        model = AttentionMIL(in_dim, hidden_dim=args.hidden_dim, dropout=args.dropout, gated=True)
    else:
        model = AttentionMIL(in_dim, hidden_dim=args.hidden_dim, dropout=args.dropout, gated=False)
    model = model.to(device)

    train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = make_loaders(
        args.manifest, args.max_tiles, args.num_workers
    )
    if train_loader is None:
        raise RuntimeError('train split is empty')

    pos = float((manifest['label'] == 1).sum())
    neg = float((manifest['label'] == 0).sum())
    auto_pw = (neg + 1.0) / (pos + 1.0)
    pos_weight = torch.tensor([args.pos_weight if args.pos_weight > 0 else auto_pw], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(args.epochs, 1))
    scaler = torch.cuda.amp.GradScaler(enabled=args.precision == 'fp16')

    print(f"[단계] 데이터 크기 -> train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")
    print(f"[단계] {device}에서 {args.epochs} epoch 학습 시작 (precision {args.precision})")

    best_auc = -1.0
    best_path = Path(args.ckpt_dir) / 'best.pt'
    Path(args.ckpt_dir).mkdir(parents=True, exist_ok=True)
    Path(args.logdir).mkdir(parents=True, exist_ok=True)

    def run_split(loader, split_name):
        if loader is None:
            return {'auc': float('nan'), 'ap': float('nan'), 'f1': float('nan'), 'acc': float('nan'), 'confusion': {}}, []
        model.eval()
        y_true, y_prob, rows = [], [], []
        with torch.inference_mode():
            for feats, coords, labels, slide_ids in loader:
                feats = feats[0].to(device)
                coords = coords[0].to(device) if coords[0] is not None else None
                labels = labels.to(device)
                logit, attn = model(feats, coords)
                prob = torch.sigmoid(logit).item()
                y_true.append(float(labels.item()))
                y_prob.append(prob)
                rows.append({'slide_id': slide_ids[0], 'prob': prob, 'label': float(labels.item())})
        metrics = compute_metrics(y_true, y_prob, threshold=args.threshold)
        return metrics, rows

    history = []
    for epoch in range(1, args.epochs + 1):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f'train epoch {epoch}', ncols=100)
        for step, (feats, coords, labels, slide_ids) in enumerate(pbar, 1):
            feats = feats[0].to(device)
            coords = coords[0].to(device) if coords[0] is not None else None
            labels = labels.to(device)
            if args.label_smoothing > 0:
                eps = args.label_smoothing
                labels = labels * (1 - eps) + 0.5 * eps
            with torch.cuda.amp.autocast(enabled=args.precision == 'fp16'):
                logit, attn = model(feats, coords)
                loss = criterion(logit.view_as(labels), labels)
                loss = loss / args.grad_accum
            scaler.scale(loss).backward()
            if step % args.grad_accum == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            running_loss += loss.item()
            pbar.set_postfix({'loss': running_loss / step})
        scheduler.step()

        val_metrics, val_rows = run_split(val_loader, 'val')
        history.append({'epoch': epoch, 'val': val_metrics, 'loss': running_loss / max(1, step)})
        print(f"epoch {epoch}: val_auc={val_metrics['auc']:.4f} f1={val_metrics['f1']:.4f}")

        if val_metrics['auc'] > best_auc:
            best_auc = val_metrics['auc']
            torch.save(
                {'model_state': model.state_dict(), 'config': vars(args), 'val_metrics': val_metrics},
                best_path,
            )
            print('=> 최고 성능 갱신, 체크포인트 저장:', best_path)

    best = torch.load(best_path, map_location=device)
    model.load_state_dict(best['model_state'])
    val_metrics, val_rows = run_split(val_loader, 'val')
    test_metrics, test_rows = run_split(test_loader, 'test')
    summary = {
        'best_val_auc': best_auc,
        'val': val_metrics,
        'test': test_metrics,
        'history': history,
    }
    log_path = Path(args.logdir) / 'mil_training.json'
    with open(log_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print('== 최종 성능 ==')
    print('  validation:', val_metrics)
    print('  test      :', test_metrics)
    print('결과 저장:', log_path)
    if test_rows:
        pd.DataFrame(test_rows).to_csv(Path(args.logdir) / 'test_preds.csv', index=False)
    if val_rows:
        pd.DataFrame(val_rows).to_csv(Path(args.logdir) / 'val_preds.csv', index=False)


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--manifest', required=True)
    p.add_argument('--ckpt_dir', default='checkpoints')
    p.add_argument('--logdir', default='logs')
    p.add_argument('--model', choices=['abmil', 'clam', 'transmil'], default='abmil')
    p.add_argument('--hidden_dim', type=int, default=256)
    p.add_argument('--depth', type=int, default=2)
    p.add_argument('--heads', type=int, default=4)
    p.add_argument('--ffn_dim', type=int, default=512)
    p.add_argument('--dropout', type=float, default=0.2)
    p.add_argument('--epochs', type=int, default=20)
    p.add_argument('--lr', type=float, default=2e-4)
    p.add_argument('--weight_decay', type=float, default=1e-4)
    p.add_argument('--pos_weight', type=float, default=-1.0)
    p.add_argument('--label_smoothing', type=float, default=0.0)
    p.add_argument('--precision', choices=['fp16', 'fp32'], default='fp16')
    p.add_argument('--grad_accum', type=int, default=1)
    p.add_argument('--max_tiles', type=int, default=None)
    p.add_argument('--num_workers', type=int, default=2)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--threshold', type=float, default=0.5)
    p.add_argument('--use_pos_enc', action='store_true')
    return p.parse_args()


if __name__ == '__main__':
    args = parse_args()
    train(args)
'''


if WRITE_TRAINER:
    cmd = (f"cat > {REMOTE_TRAIN_SCRIPT} <<'PY'\n"           f"{TRAIN_SCRIPT}\n"           "PY\n"           f"chmod +x {REMOTE_TRAIN_SCRIPT}")
    run_ssh(cmd)
    print('원격 학습 스크립트 저장 완료 ->', REMOTE_TRAIN_SCRIPT)
else:
    print('Training script generation skipped (set WRITE_TRAINER=True)')


[ssh] $ cat > /workspace/data/work/train_mil.py <<'PY'

import argparse, json, os, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score, confusion_matrix

FEATURE_KEYS = ['features', 'feats', 'embedding', 'embeddings', 'feat']
COORD_KEYS = ['coords', 'coord', 'locs', 'locations']

def load_feats_coords(data):
    import re
    # 우선순위: 명시적인 키 -> last_layer_embed -> layer_<n>_embed 중 가장 높은 레이어
    feature_keys = ['features', 'feats', 'embedding', 'embeddings', 'feat', 'last_layer_embed']
    feats = None
    for k in feature_keys:
        if k in data:
            feats = data[k]
            break
    if feats is None:
        layer_keys = [k for k in data.keys() if re.match(r'^layer_\d+_embed$', k)]
        if layer_keys:
            best = sorted(layer_keys, key=lambda x: int(re.findall(r'\d+', x)[0]))[-1]
   

## 6. 학습 실행 커맨드 준비/실행
- 모델 선택: `abmil`(기본), `clam`(gated attention), `transmil`(transformer)
- `MAX_TILES_PER_SLIDE`/`grad_accum`/`precision` 등 조정 후 실행
- `RUN_TRAINING=True`로 토글 시 즉시 RunPod에서 학습 시작


In [7]:
train_cmd = f"{REMOTE_VENV}/bin/python {REMOTE_TRAIN_SCRIPT} "     f"--manifest {REMOTE_MANIFEST} "     f"--ckpt_dir {REMOTE_CHECKPOINT} --logdir {REMOTE_LOG} "     f"--model transmil --use_pos_enc --max_tiles {MAX_TILES_PER_SLIDE} "     f"--epochs 20 --lr 2e-4 --grad_accum 2 --precision fp16 --hidden_dim 256 --depth 2 --heads 4 --ffn_dim 768"

print('[안내] 학습 실행 커맨드:', train_cmd)

if RUN_TRAINING:
    run_ssh(train_cmd)
else:
    print('학습 실행 건너뜀 (RUN_TRAINING=True 로 설정 시 실행)')


[안내] 학습 실행 커맨드: ~/venv_gigapath/bin/python /workspace/data/work/train_mil.py --manifest /workspace/data/work/manifest.csv --ckpt_dir /workspace/data/work/checkpoints --logdir /workspace/data/logs --model transmil --use_pos_enc --max_tiles 8000 --epochs 20 --lr 2e-4 --grad_accum 2 --precision fp16 --hidden_dim 256 --depth 2 --heads 4 --ffn_dim 768
[ssh] $ ~/venv_gigapath/bin/python /workspace/data/work/train_mil.py --manifest /workspace/data/work/manifest.csv --ckpt_dir /workspace/data/work/checkpoints --logdir /workspace/data/logs --model transmil --use_pos_enc --max_tiles 8000 --epochs 20 --lr 2e-4 --grad_accum 2 --precision fp16 --hidden_dim 256 --depth 2 --heads 4 --ffn_dim 768
[단계] manifest 로드 (/workspace/data/work/manifest.csv) -> 총 48 슬라이드
       split 분포: {'train': 41, 'val': 7}
       라벨 분포: {0: 31, 1: 17}
[단계] 데이터 크기 -> train 41 | val 7 | test 0
[단계] cuda에서 20 epoch 학습 시작 (precision fp16)
epoch 1: val_auc=0.7000 f1=0.4444
=> 최고 성능 갱신, 체크포인트 저장: /workspace/data/work/checkpoints

## 7. 성능 평가/로그 확인

In [8]:
# 원격 로그 파일에서 성능/예측을 요약합니다.
python_cmd = f"{REMOTE_VENV}/bin/python - <<'PY'\nimport json, pandas as pd, pathlib\nlog_path = pathlib.Path('{REMOTE_LOG}') / 'mil_training.json'\nval_csv = pathlib.Path('{REMOTE_LOG}') / 'val_preds.csv'\ntest_csv = pathlib.Path('{REMOTE_LOG}') / 'test_preds.csv'\nif not log_path.exists():\n    print('[오류] 로그 파일이 없습니다:', log_path)\n    raise SystemExit(1)\nwith open(log_path) as f: data = json.load(f)\nprint('[로그]', log_path)\nprint(' best_val_auc:', data.get('best_val_auc'))\nprint(' val metrics :', data.get('val'))\nprint(' test metrics:', data.get('test'))\nif val_csv.exists():\n    df = pd.read_csv(val_csv)\n    print('\\n[val preds head]')\n    print(df.head())\nif test_csv.exists():\n    df = pd.read_csv(test_csv)\n    print('\\n[test preds head]')\n    print(df.head())\nPY"

print('[단계] 원격 성능 로그 확인 명령:')
print(python_cmd)
run_ssh(python_cmd)


[단계] 원격 성능 로그 확인 명령:
~/venv_gigapath/bin/python - <<'PY'
import json, pandas as pd, pathlib
log_path = pathlib.Path('/workspace/data/logs') / 'mil_training.json'
val_csv = pathlib.Path('/workspace/data/logs') / 'val_preds.csv'
test_csv = pathlib.Path('/workspace/data/logs') / 'test_preds.csv'
if not log_path.exists():
    print('[오류] 로그 파일이 없습니다:', log_path)
    raise SystemExit(1)
with open(log_path) as f: data = json.load(f)
print('[로그]', log_path)
print(' best_val_auc:', data.get('best_val_auc'))
print(' val metrics :', data.get('val'))
print(' test metrics:', data.get('test'))
if val_csv.exists():
    df = pd.read_csv(val_csv)
    print('\n[val preds head]')
    print(df.head())
if test_csv.exists():
    df = pd.read_csv(test_csv)
    print('\n[test preds head]')
    print(df.head())
PY
[ssh] $ ~/venv_gigapath/bin/python - <<'PY'
import json, pandas as pd, pathlib
log_path = pathlib.Path('/workspace/data/logs') / 'mil_training.json'
val_csv = pathlib.Path('/workspace/data/logs') / 

0